In [6]:
# Cell 1: Import dependencies and resolve VGG full-complex evaluation paths
import json
import os
import re
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import yaml
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import load_model

notebook_dir = Path().resolve()
if notebook_dir.name == 'notebooks':
    project_root = notebook_dir.parent
elif (notebook_dir / 'notebooks').exists() and (notebook_dir / 'src').exists():
    project_root = notebook_dir
else:
    project_root = notebook_dir

cfg_path = project_root / 'configs' / 'local_data_paths.yaml'
if cfg_path.exists():
    cfg = yaml.safe_load(cfg_path.read_text()) or {}
    dcfg = cfg.get('datasets', {}).get('noisy_drone_rf_v2', {}) or {}
    data_dir = Path(dcfg.get('data_dir', '/scratch/rameyjm7/datasets/NoisyDroneRFv2'))
else:
    data_dir = Path('/scratch/rameyjm7/datasets/NoisyDroneRFv2')

model_dir = project_root / 'models' / 'noisy_drone_rf_v2'
vgg_best_path = model_dir / 'noisy_drone_rf_v2_vgg_full_complex_spectrogram_best.keras'
vgg_final_path = model_dir / 'noisy_drone_rf_v2_vgg_full_complex_spectrogram_final.keras'
model_path = vgg_final_path if vgg_final_path.exists() else vgg_best_path

outputs_dir = project_root / 'outputs' / 'noisy_drone_rf_v2_eval'
outputs_dir.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 1961
MAX_IQ_SAMPLES = int(os.getenv('NOISY_DRONE_MAX_IQ_SAMPLES', '1048576'))
MIN_SNR_DB = float(os.getenv('NOISY_DRONE_MIN_SNR_DB', '-6'))
DATA_FRACTION = float(os.getenv('NOISY_DRONE_DATA_FRACTION', '0.25'))
BATCH_SIZE = int(os.getenv('NOISY_DRONE_BATCH_SIZE', '8'))
SPEC_NFFT = int(os.getenv('NOISY_DRONE_SPEC_NFFT', '1024'))
SPEC_HOP = int(os.getenv('NOISY_DRONE_SPEC_HOP', '1024'))
SPEC_TIME_BINS = int(os.getenv('NOISY_DRONE_SPEC_TIME_BINS', '1024'))
SPEC_EVAL_WINDOWS = int(os.getenv('NOISY_DRONE_SPEC_EVAL_WINDOWS', '1'))
BURST_SMOOTH_SAMPLES = int(os.getenv('NOISY_DRONE_BURST_SMOOTH_SAMPLES', '512'))
BALANCED_EVAL = os.getenv('NOISY_DRONE_BALANCED_EVAL', '1').lower() not in {'0', 'false', 'no'}

cache_dir = Path(os.getenv(
    'NOISY_DRONE_SPEC_CACHE_DIR',
    '/scratch/rameyjm7/ML-wireless-signal-classification/cache/noisy_drone_rf_v2/spectrogram_full_complex_cache',
))
cache_dir.mkdir(parents=True, exist_ok=True)

print('Noisy Drone RF v2 data:', data_dir)
print('Noisy Drone RF v2 VGG model:', model_path)
print('Outputs dir:', outputs_dir)
print('Spectrogram cache:', cache_dir)
assert data_dir.exists(), f'Missing NoisyDroneRFv2 directory: {data_dir}'
assert model_path.exists(), f'Missing VGG model: {model_path}'


Noisy Drone RF v2 data: /scratch/rameyjm7/datasets/NoisyDroneRFv2
Noisy Drone RF v2 VGG model: /home/rameyjm7/workspace/ML-wireless-signal-classification/models/noisy_drone_rf_v2/noisy_drone_rf_v2_vgg_full_complex_spectrogram_best.keras
Outputs dir: /home/rameyjm7/workspace/ML-wireless-signal-classification/outputs/noisy_drone_rf_v2_eval
Spectrogram cache: /scratch/rameyjm7/ML-wireless-signal-classification/cache/noisy_drone_rf_v2/spectrogram_full_complex_cache


In [7]:
# Cell 2: Build the Noisy Drone RF v2 eval manifest and full-complex spectrogram helpers
sample_re = re.compile(r'IQdata_sample(?P<sample>\d+)_target(?P<target>-?\d+)_snr(?P<snr>-?\d+)\.pt$')
pt_files = sorted(data_dir.rglob('IQdata_sample*_target*_snr*.pt'))
assert pt_files, f'No matching .pt files found under {data_dir}'

class_stats_path = data_dir / 'class_stats.csv'
class_stats = pd.read_csv(class_stats_path) if class_stats_path.exists() else None
rows = []
for filepath in pt_files:
    match = sample_re.search(filepath.name)
    if not match:
        continue
    rows.append({
        'filepath': str(filepath),
        'sample_id': int(match.group('sample')),
        'target_raw': int(match.group('target')),
        'snr': int(match.group('snr')),
    })

data_df = pd.DataFrame(rows).sort_values('sample_id').reset_index(drop=True)
full_sample_count = len(data_df)
if MIN_SNR_DB > -999:
    data_df = data_df[data_df['snr'] >= MIN_SNR_DB].reset_index(drop=True)
if DATA_FRACTION < 1.0:
    data_df = (
        data_df.groupby('target_raw', group_keys=False)
        .sample(frac=DATA_FRACTION, random_state=RANDOM_STATE)
        .sort_values('sample_id')
        .reset_index(drop=True)
    )

classes = np.array(sorted(data_df['target_raw'].unique()), dtype=np.int64)
class_to_index = {int(c): idx for idx, c in enumerate(classes)}
if class_stats is not None and {'class_int', 'class'}.issubset(class_stats.columns):
    class_name_lookup = dict(zip(class_stats['class_int'].astype(int), class_stats['class'].astype(str)))
else:
    class_name_lookup = {int(c): f'target_{int(c)}' for c in classes}
label_names = [class_name_lookup.get(int(c), f'target_{int(c)}') for c in classes]
data_df['label_idx'] = data_df['target_raw'].map(class_to_index).astype(np.int64)

print('Samples selected:', len(data_df), '/', full_sample_count)
print('Class names:', label_names)
print('Class counts:', data_df['label_idx'].value_counts().sort_index().to_dict())


def load_pt_iq(filepath):
    obj = torch.load(filepath, map_location='cpu')

    def extract_iq(value):
        if isinstance(value, dict):
            preferred_keys = (
                'x_iq', 'iq', 'IQ', 'x', 'X', 'data', 'samples',
                'signal', 'waveform', 'input', 'features', 'arr', 'array',
            )
            for key in preferred_keys:
                if key in value:
                    return extract_iq(value[key])
            for key, item in value.items():
                if hasattr(item, 'detach') or isinstance(item, (np.ndarray, list, tuple, dict)):
                    try:
                        return extract_iq(item)
                    except (TypeError, ValueError, KeyError):
                        continue
            raise KeyError(f'No IQ-like payload found in {filepath}; keys={list(value.keys())}')
        if isinstance(value, (tuple, list)):
            if not value:
                raise ValueError(f'Empty sequence payload in {filepath}')
            return extract_iq(value[0])
        return value

    obj = extract_iq(obj)
    arr = obj.detach().cpu().numpy() if hasattr(obj, 'detach') else np.asarray(obj)
    arr = np.squeeze(arr)
    if arr.ndim == 1:
        if np.iscomplexobj(arr):
            arr = np.stack([arr.real, arr.imag], axis=-1)
        else:
            if arr.size % 2 != 0:
                arr = arr[:-1]
            arr = arr.reshape(-1, 2)
    elif arr.ndim == 2:
        if arr.shape[0] == 2 and arr.shape[1] != 2:
            arr = arr.T
        elif arr.shape[-1] != 2:
            raise ValueError(f'Expected IQ tensor with final dimension 2, got {arr.shape} in {filepath}')
    else:
        if arr.shape[-1] == 2:
            arr = arr.reshape(-1, 2)
        elif arr.shape[0] == 2:
            arr = np.moveaxis(arr, 0, -1).reshape(-1, 2)
        else:
            raise ValueError(f'Expected IQ tensor with two channels, got {arr.shape} in {filepath}')
    return np.asarray(arr, dtype=np.float32)

raw_sample_len = load_pt_iq(data_df.iloc[0]['filepath']).shape[0]
SAMPLE_LEN = min(raw_sample_len, MAX_IQ_SAMPLES)
num_classes = len(classes)
input_shape = (SPEC_NFFT, SPEC_TIME_BINS, 2)
print('Raw sample length:', raw_sample_len)
print('VGG full-complex input shape:', input_shape)


def find_burst_start(x, window_len):
    if x.shape[0] <= window_len:
        return 0
    power = np.mean(np.square(x.astype(np.float32)), axis=1)
    smooth_len = max(1, min(BURST_SMOOTH_SAMPLES, power.shape[0] // 8))
    if smooth_len > 1:
        kernel = np.ones(smooth_len, dtype=np.float32) / float(smooth_len)
        power = np.convolve(power, kernel, mode='same')
    center = int(np.argmax(power))
    return int(np.clip(center - window_len // 2, 0, x.shape[0] - window_len))


def normalize_iq_window(iq):
    iq = np.asarray(iq[:, :2], dtype=np.float32)
    iq = iq - np.mean(iq, axis=0, keepdims=True)
    scale = np.sqrt(np.mean(np.square(iq)) + 1e-8)
    return iq / scale


def resize_time_axis(arr, target_bins):
    if arr.shape[1] == target_bins:
        return arr
    src_x = np.linspace(0.0, 1.0, arr.shape[1], dtype=np.float32)
    dst_x = np.linspace(0.0, 1.0, target_bins, dtype=np.float32)
    return np.stack([np.interp(dst_x, src_x, row).astype(np.float32) for row in arr], axis=0)


def iq_window_to_spectrogram(iq, snr_value):
    iq = normalize_iq_window(iq[:, :2])
    complex_iq = iq[:, 0].astype(np.float32) + 1j * iq[:, 1].astype(np.float32)
    if len(complex_iq) < SPEC_NFFT:
        complex_iq = np.pad(complex_iq, (0, SPEC_NFFT - len(complex_iq)), mode='constant')
    starts = np.arange(0, len(complex_iq) - SPEC_NFFT + 1, SPEC_HOP)
    if starts.size == 0:
        starts = np.array([0])
    window = np.hanning(SPEC_NFFT).astype(np.float32)
    frames = np.stack([complex_iq[s:s + SPEC_NFFT] * window for s in starts], axis=0)
    fft_complex = np.fft.fftshift(np.fft.fft(frames, n=SPEC_NFFT, axis=1), axes=1).T
    fft_complex = fft_complex / float(SPEC_NFFT)
    real_part = resize_time_axis(fft_complex.real.astype(np.float32), SPEC_TIME_BINS)
    imag_part = resize_time_axis(fft_complex.imag.astype(np.float32), SPEC_TIME_BINS)
    spec = np.stack([real_part, imag_part], axis=-1).astype(np.float32)
    spec = spec / (np.std(spec) + 1e-6)
    return np.clip(spec, -6.0, 6.0).astype(np.float32)


def spectrogram_cache_path(filepath, snr_value):
    src = Path(str(filepath))
    name = f'{src.stem}_full_complex_len{SAMPLE_LEN}_nfft{SPEC_NFFT}_hop{SPEC_HOP}_tb{SPEC_TIME_BINS}_snr{int(float(snr_value))}.npz'
    return cache_dir / name


def safe_load_cached_spectrogram(cache_path):
    try:
        with np.load(cache_path) as data:
            x = data['x'].astype(np.float32)
        if x.shape != input_shape:
            raise ValueError(f'cached shape {x.shape} != expected {input_shape}')
        if not np.isfinite(x).all():
            raise ValueError('cached spectrogram contains NaN or Inf')
        return x
    except (EOFError, OSError, ValueError, KeyError, zipfile.BadZipFile) as exc:
        print(f'Removing corrupt spectrogram cache: {cache_path} ({type(exc).__name__}: {exc})')
        try:
            Path(cache_path).unlink(missing_ok=True)
        except OSError as unlink_exc:
            print(f'Warning: could not remove corrupt cache {cache_path}: {unlink_exc}')
        return None


def write_spectrogram_cache_atomic(cache_path, x):
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = cache_path.with_suffix(cache_path.suffix + f'.{os.getpid()}.tmp')
    with tmp_path.open('wb') as handle:
        np.savez_compressed(handle, x=x.astype(np.float32))
    tmp_path.replace(cache_path)


def prepare_spectrogram(filepath, snr_value):
    cache_path = spectrogram_cache_path(filepath, snr_value)
    if cache_path.exists():
        cached = safe_load_cached_spectrogram(cache_path)
        if cached is not None:
            return cached
    x_full = load_pt_iq(filepath)
    if x_full.shape[0] < SAMPLE_LEN:
        x_full = np.pad(x_full, ((0, SAMPLE_LEN - x_full.shape[0]), (0, 0)), mode='constant')
    start = find_burst_start(x_full, SAMPLE_LEN)
    x = iq_window_to_spectrogram(x_full[start:start + SAMPLE_LEN, :2], snr_value)
    write_spectrogram_cache_atomic(cache_path, x)
    return x.astype(np.float32)


def prepare_spectrogram_windows(filepath, snr_value, n_windows=None):
    n_windows = SPEC_EVAL_WINDOWS if n_windows is None else int(n_windows)
    if n_windows <= 1:
        return prepare_spectrogram(filepath, snr_value)[None, ...]
    x_full = load_pt_iq(filepath)
    if x_full.shape[0] < SAMPLE_LEN:
        x_full = np.pad(x_full, ((0, SAMPLE_LEN - x_full.shape[0]), (0, 0)), mode='constant')
    center_start = find_burst_start(x_full, SAMPLE_LEN)
    if x_full.shape[0] <= SAMPLE_LEN:
        starts = [0]
    else:
        stride = max(1, SAMPLE_LEN // max(4, n_windows + 1))
        offsets = (np.arange(n_windows) - ((n_windows - 1) / 2.0)) * stride
        starts = [int(np.clip(center_start + off, 0, x_full.shape[0] - SAMPLE_LEN)) for off in offsets]
        starts = sorted(set(starts))
    return np.stack([iq_window_to_spectrogram(x_full[start:start + SAMPLE_LEN, :2], snr_value) for start in starts], axis=0).astype(np.float32)


def make_balanced_eval_df(split_df):
    target_n = int(split_df['label_idx'].value_counts().min())
    return (
        split_df.groupby('label_idx', group_keys=False)
        .sample(n=target_n, random_state=RANDOM_STATE)
        .sample(frac=1.0, random_state=RANDOM_STATE)
        .reset_index(drop=True)
    )

idx = np.arange(len(data_df))
train_idx, test_idx = train_test_split(idx, test_size=0.20, random_state=RANDOM_STATE, stratify=data_df['label_idx'])
_, val_idx = train_test_split(train_idx, test_size=0.20, random_state=RANDOM_STATE, stratify=data_df.iloc[train_idx]['label_idx'])
test_df = data_df.iloc[test_idx].reset_index(drop=True)
balanced_test_df = make_balanced_eval_df(test_df) if BALANCED_EVAL else None
print('Test natural counts:', test_df['label_idx'].value_counts().sort_index().to_dict())
if balanced_test_df is not None:
    print('Test balanced counts:', balanced_test_df['label_idx'].value_counts().sort_index().to_dict())


Samples selected: 3242 / 17744
Class names: ['DJI', 'FutabaT14', 'FutabaT7', 'Graupner', 'Noise', 'Taranis', 'Turnigy']
Class counts: {0: 236, 1: 635, 2: 148, 3: 144, 4: 1622, 5: 300, 6: 157}
Raw sample length: 1048576
VGG full-complex input shape: (1024, 1024, 2)
Test natural counts: {0: 47, 1: 127, 2: 30, 3: 29, 4: 325, 5: 60, 6: 31}
Test balanced counts: {0: 29, 1: 29, 2: 29, 3: 29, 4: 29, 5: 29, 6: 29}


In [ ]:
# Cell 3: Evaluate the canonical VGG full-complex spectrogram model
vgg_eval_model = load_model(model_path, compile=False)
print('Loaded model:', model_path)
print('Model input shape:', vgg_eval_model.input_shape)


def predict_vgg(split_df, eval_windows=None, name='eval'):
    probs = []
    total = len(split_df)
    for i, row in enumerate(split_df.itertuples(index=False), start=1):
        x = prepare_spectrogram_windows(row.filepath, row.snr, n_windows=eval_windows)
        window_probs = vgg_eval_model.predict(x, batch_size=BATCH_SIZE, verbose=0)
        probs.append(window_probs.mean(axis=0))
        if i <= 3 or i % 100 == 0 or i == total:
            print(f'{name}: {i}/{total}', flush=True)
    return split_df.copy(), np.asarray(probs, dtype=np.float32)

natural_eval_df, natural_probs = predict_vgg(test_df, eval_windows=SPEC_EVAL_WINDOWS, name='natural')
y_true = natural_eval_df['label_idx'].to_numpy(dtype=np.int64)
y_pred = natural_probs.argmax(axis=1)

natural_metrics = {
    'accuracy': float(accuracy_score(y_true, y_pred)),
    'macro_f1': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
    'weighted_f1': float(f1_score(y_true, y_pred, average='weighted', zero_division=0)),
}
print(f'Noisy Drone RF v2 VGG natural report ({SPEC_EVAL_WINDOWS} eval windows)')
print(natural_metrics)
print(classification_report(y_true, y_pred, target_names=label_names, zero_division=0))

report_csv = outputs_dir / '44_noisy_drone_rf_v2_vgg_full_complex_spectrogram_classification_report.csv'
pd.DataFrame(classification_report(y_true, y_pred, target_names=label_names, zero_division=0, output_dict=True)).transpose().to_csv(report_csv)
print('Saved classification report:', report_csv)

balanced_metrics = None
if balanced_test_df is not None:
    balanced_eval_df, balanced_probs = predict_vgg(balanced_test_df, eval_windows=SPEC_EVAL_WINDOWS, name='balanced')
    y_balanced_true = balanced_eval_df['label_idx'].to_numpy(dtype=np.int64)
    y_balanced_pred = balanced_probs.argmax(axis=1)
    balanced_metrics = {
        'balanced_accuracy': float(accuracy_score(y_balanced_true, y_balanced_pred)),
        'balanced_macro_f1': float(f1_score(y_balanced_true, y_balanced_pred, average='macro', zero_division=0)),
        'balanced_weighted_f1': float(f1_score(y_balanced_true, y_balanced_pred, average='weighted', zero_division=0)),
    }
    print(f'Noisy Drone RF v2 VGG balanced report ({SPEC_EVAL_WINDOWS} eval windows)')
    print(balanced_metrics)
    print(classification_report(y_balanced_true, y_balanced_pred, target_names=label_names, zero_division=0))

metrics = {
    'model': 'noisy_drone_rf_v2_vgg_full_complex_spectrogram',
    'model_path': str(model_path),
    'eval_windows': int(SPEC_EVAL_WINDOWS),
    'sample_len': int(SAMPLE_LEN),
    'min_snr_db': float(MIN_SNR_DB),
    'data_fraction': float(DATA_FRACTION),
    'num_classes': int(num_classes),
    'test_samples': int(len(test_df)),
    'label_names': label_names,
    **natural_metrics,
}
if balanced_metrics is not None:
    metrics.update(balanced_metrics)
metrics_path = outputs_dir / '44_noisy_drone_rf_v2_vgg_full_complex_spectrogram_metrics.json'
metrics_path.write_text(json.dumps(metrics, indent=2), encoding='utf-8')
print('Saved metrics:', metrics_path)


Loaded model: /home/rameyjm7/workspace/ML-wireless-signal-classification/models/noisy_drone_rf_v2/noisy_drone_rf_v2_vgg_full_complex_spectrogram_best.keras
Model input shape: (None, 1024, 1024, 2)


I0000 00:00:1783005779.706224 1035911 service.cc:153] XLA service 0x15332be51480 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1783005779.706254 1035911 service.cc:161]   StreamExecutor [0]: NVIDIA A100-SXM4-80GB, Compute Capability 8.0 (Driver: 13.0.0; Runtime: 12.8.0; Toolkit: 12.5.0; DNN: 9.10.2)
I0000 00:00:1783005779.955945 1035911 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.


natural: 1/649
natural: 2/649


I0000 00:00:1783005784.051734 1035911 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


natural: 3/649
natural: 100/649


In [ ]:
# Cell 4: Plot natural and balanced confusion matrices for the VGG full-complex model
cm = confusion_matrix(y_true, y_pred, labels=np.arange(num_classes))
plt.figure(figsize=(12, 10))
sns.heatmap(cm, cmap='Blues', xticklabels=label_names, yticklabels=label_names)
plt.title('Noisy Drone RF v2 VGG Full-Complex Spectrogram - Natural Test Distribution')
plt.xlabel('Predicted label')
plt.ylabel('True label')
plt.tight_layout()
cm_png = outputs_dir / '44_noisy_drone_rf_v2_vgg_full_complex_spectrogram_confusion_matrix.png'
plt.savefig(cm_png, dpi=180)
print('Saved confusion matrix:', cm_png)
plt.show()

if balanced_test_df is not None:
    balanced_cm = confusion_matrix(y_balanced_true, y_balanced_pred, labels=np.arange(num_classes))
    plt.figure(figsize=(12, 10))
    sns.heatmap(balanced_cm, cmap='Blues', xticklabels=label_names, yticklabels=label_names)
    plt.title('Noisy Drone RF v2 VGG Full-Complex Spectrogram - Balanced Test Distribution')
    plt.xlabel('Predicted label')
    plt.ylabel('True label')
    plt.tight_layout()
    balanced_cm_png = outputs_dir / '44_noisy_drone_rf_v2_vgg_full_complex_spectrogram_balanced_confusion_matrix.png'
    plt.savefig(balanced_cm_png, dpi=180)
    print('Saved balanced confusion matrix:', balanced_cm_png)
    plt.show()


In [ ]:
# Cell 5: Plot and save VGG full-complex accuracy across SNR, overall and per class
snr_eval_df = natural_eval_df.copy()
snr_eval_df['y_true'] = y_true
snr_eval_df['y_pred'] = y_pred
snr_eval_df['correct'] = snr_eval_df['y_true'] == snr_eval_df['y_pred']

snr_results_path = outputs_dir / '44_noisy_drone_rf_v2_vgg_full_complex_snr_results.csv'
snr_eval_df.to_csv(snr_results_path, index=False)
print('Saved SNR-level predictions:', snr_results_path)

overall_snr = (
    snr_eval_df.groupby('snr')
    .agg(accuracy=('correct', 'mean'), samples=('correct', 'size'))
    .reset_index()
    .sort_values('snr')
)
overall_snr['accuracy_percent'] = overall_snr['accuracy'] * 100.0
overall_path = outputs_dir / '44_noisy_drone_rf_v2_vgg_full_complex_accuracy_vs_snr.csv'
overall_snr.to_csv(overall_path, index=False)
print('Saved overall SNR accuracy:', overall_path)

per_class_rows = []
for (snr_value, label_idx), group in snr_eval_df.groupby(['snr', 'y_true']):
    per_class_rows.append({
        'snr': int(snr_value),
        'label_idx': int(label_idx),
        'label': label_names[int(label_idx)],
        'accuracy': float(group['correct'].mean()),
        'accuracy_percent': float(group['correct'].mean() * 100.0),
        'samples': int(len(group)),
    })
per_class_snr = pd.DataFrame(per_class_rows).sort_values(['label_idx', 'snr']).reset_index(drop=True)
per_class_path = outputs_dir / '44_noisy_drone_rf_v2_vgg_full_complex_accuracy_vs_snr_per_class.csv'
per_class_snr.to_csv(per_class_path, index=False)
print('Saved per-class SNR accuracy:', per_class_path)

fig, ax = plt.subplots(figsize=(12, 7))
ax.plot(overall_snr['snr'], overall_snr['accuracy_percent'], marker='o')
for _, row in overall_snr.iterrows():
    ax.annotate(f"n={int(row['samples'])}", (row['snr'], row['accuracy_percent']), textcoords='offset points', xytext=(0, 8), ha='center', fontsize=8)
ax.set_title('Noisy Drone RF v2 VGG Full-Complex Spectrogram - Accuracy vs. SNR')
ax.set_xlabel('SNR (dB)')
ax.set_ylabel('Accuracy (%)')
ax.set_ylim(0, 105)
ax.grid(True, alpha=0.3)
plt.tight_layout()
overall_plot_path = outputs_dir / '44_noisy_drone_rf_v2_vgg_full_complex_accuracy_vs_snr.png'
plt.savefig(overall_plot_path, dpi=180)
print('Saved:', overall_plot_path)
plt.show()

fig, ax = plt.subplots(figsize=(14, 8))
for label_idx, group in per_class_snr.groupby('label_idx'):
    ax.plot(group['snr'], group['accuracy_percent'], marker='o', linewidth=1.5, label=label_names[int(label_idx)])
ax.set_title('Noisy Drone RF v2 VGG Full-Complex Spectrogram - Accuracy vs. SNR per Class')
ax.set_xlabel('SNR (dB)')
ax.set_ylabel('Accuracy (%)')
ax.set_ylim(0, 105)
ax.grid(True, alpha=0.3)
ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=9)
plt.tight_layout()
per_class_plot_path = outputs_dir / '44_noisy_drone_rf_v2_vgg_full_complex_accuracy_vs_snr_per_class.png'
plt.savefig(per_class_plot_path, dpi=180)
print('Saved:', per_class_plot_path)
plt.show()
